# TRABALHO AUTÓNOMO 2

O objetivo deste trabalho é completar o programa do agente para explorar o mundo Wumpus tentando obter a máxima performance antes de o agente morrer ou sair da caverna com o ouro e eventualmente matar o Wumpus


Primeiro vamos importar os pacotes necessários:

In [26]:
from ipythonblocks import BlockGrid
from agents import *
from logic4e import *

## Coisas que existem no mundo

Depois vamos definir as coisas que existem no mundo Wumpus

In [27]:
class Ouro(Thing):
    pass

class Parede(Thing):
    pass

class Brilho(Thing):
    pass

class Poco(Thing):
    pass

class Brisa(Thing):
    pass

class Flecha(Thing):
    pass

class Wumpus(Agent):
    gritou = False
    pass

class Fedor(Thing):
    pass

class Grito(Agent):
    pass

## O Ambiente

Vamos defenir a classe do ambiente:

In [28]:
class WumpusEnvironment(XYEnvironment):

    probabilidade = 0.1  # probabilidade de espalhar poços pelo ambiente

    ''' A caverna quando for de 4x4 deve ter mais 2 extra para as paredes'''
    def __init__(self, agente, dim=4):
        super().__init__(dim+2, dim+2)
        self.init_world(agente)

    
    def init_world(self,agente):
        '''Espalha coisas pela caverna de acordo com a probabilidade definida'''

        ''' atualiza começo e final'''

        "Paredes"
        
        for x in range(self.width-1):
            self.add_thing(Parede(), (x, 0),True)
            self.add_thing(Parede(), (x, self.height - 1),True)
        for y in range(0, self.height):
            self.add_thing(Parede(), (0, y),True)
            self.add_thing(Parede(), (self.width - 1, y),True)

        self.x_start, self.y_start = (1, 1)
        self.x_end, self.y_end = (self.width - 1, self.height - 1)

        "Poços"
        for x in range(self.x_start, self.x_end):
            for y in range(self.y_start, self.y_end):
                if random.random() < self.probabilidade:
                    self.add_thing(Poco(), (x, y), True)
                    self.add_thing(Brisa(), (x - 1, y), True)
                    self.add_thing(Brisa(), (x, y - 1), True)
                    self.add_thing(Brisa(), (x + 1, y), True)
                    self.add_thing(Brisa(), (x, y + 1), True)

        "Wumpus"
        w_x, w_y = self.random_location_inbounds(exclude=(1, 1))
        self.add_thing(Wumpus(lambda x: ""), (w_x, w_y), True)
        self.add_thing(Fedor(), (w_x - 1, w_y), True)
        self.add_thing(Fedor(), (w_x + 1, w_y), True)
        self.add_thing(Fedor(), (w_x, w_y - 1), True)
        self.add_thing(Fedor(), (w_x, w_y + 1), True)

        "Ouro"
        self.add_thing(Ouro(), self.random_location_inbounds(exclude=(1, 1)), True)

        "Agente"
        self.add_thing(agente, (1, 1), True)

    def get_world(self, show_walls=True):
        """Devolve as coisas que existem no mundo"""
        result = []
        x_start, y_start = (0, 0) if show_walls else (1, 1)
        
        if show_walls:
            x_end, y_end = self.width, self.height
        else:
            x_end, y_end = self.width - 1, self.height - 1

        for y in range(y_start, y_end):
            row = []
            for x in range(x_start, x_end):
                row.append(self.list_things_at((x, y)))
            result.append(row)
        return result

    def percepts_from(self, agent, location, tclass=Thing):
        """ Devolve as perceções para cada localização definida e atualiza alguns items com as perceções"""
        thing_percepts = {
            Ouro: Brilho(),
            Parede: Bump(),
            Wumpus: Fedor(),
            Poco: Brisa()}

        """ O agente não é perceção de si próprio"""
        thing_percepts[agent.__class__] = None

        """ O ouro no quarto brilha"""
        if location != agent.location:
            thing_percepts[Ouro] = None

        '''junta as perceções'''
        result = [thing_percepts.get(thing.__class__, thing) for thing in self.things
                  if thing.location == location and isinstance(thing, tclass)]
        
        return result if len(result) else [None]

    def percept(self, agent):
        ''' retorna as coisas nos quartos adjacentes (não diagonais)
        no formato : [quarto à esquerda, quarto à direita, quarto em cima, quarto em baixo, na localização atual]'''
        (x, y) = agent.location
        result = []
        result.append(self.percepts_from(agent, (x - 1, y)))
        result.append(self.percepts_from(agent, (x + 1, y)))
        result.append(self.percepts_from(agent, (x, y - 1)))
        result.append(self.percepts_from(agent, (x, y + 1)))
        result.append(self.percepts_from(agent, (x, y)))

        '''Quando o wumpos morre ele grita'''
        wumpus = [thing for thing in self.things if isinstance(thing, Wumpus)]
        if len(wumpus) and not wumpus[0].alive and not wumpus[0].gritou:
            result[-1].append(Grito())
            wumpus[0].gritou = True

        return result

    def add_thing(self, thing, location=None, exclude_duplicate_class=False):
        """Sobrepõe add_thing para não reiniciar a performance do agente Explorador."""
        perf = getattr(thing, 'performance', None)
        super().add_thing(thing, location, exclude_duplicate_class)
        if isinstance(thing, Explorador) and perf is not None:
            thing.performance = perf  # restaurar performance após add_thing

    def execute_action(self, agent, action):
        """Modifica o estado do ambiente com base nas ações do agente.
        Atualiza a performance do agente"""

        if isinstance(agent, Explorador) and self.in_danger(agent):
            return
            
        agent.bump = False
        if action in ['direita', 'esquerda', 'frente', 'pega']:
            agent.executa_acao(action)
            if action == 'frente':
                self.add_thing(agent, agent.location, True)
            elif action == 'pega':
                ouros = [t for t in self.list_things_at(agent.location) if isinstance(t, Ouro)]
                if ouros:
                    agent.holding.append(ouros[0])
                    self.delete_thing(ouros[0])
            agent.performance -= 1
        elif action == 'sair':
            if agent.location == (1, 1):  
                agent.performance += 1000 if any(isinstance(t, Ouro) for t in agent.holding) else 0
                self.delete_thing(agent)
        elif action == 'dispara':
            """A flecha atravessa diretamente na direção corrente do agente"""
            if agent.has_arrow:
                local_flecha = agent.quadrado_a_seguir(agent.location)
                while self.is_inbounds(local_flecha):
                    wumpus = [thing for thing in self.list_things_at(local_flecha) if isinstance(thing, Wumpus)]
                    if len(wumpus):
                        wumpus[0].alive = False
                        break
                    local_flecha = agent.quadrado_a_seguir(local_flecha)
                agent.has_arrow = False

    def in_danger(self, agent):
        """Verifica se o agente está em perigo, se sim o agente morre"""
        for thing in self.list_things_at(agent.location):
            if isinstance(thing, Poco) or (isinstance(thing, Wumpus) and thing.alive):
                agent.alive = False
                agent.performance -= 1000
                agent.morto_por = thing.__class__.__name__
                return True
        return False

    def is_done(self):
        """O Jogo finaliza se o agente é morto ou se sobe no quarto [1,1]"""
        explorador = [agent for agent in self.agents if isinstance(agent, Explorador)]
        if len(explorador):
            if explorador[0].alive:
                return False
            else:
                print("Morto por {} [-1000].".format(explorador[0].morto_por))
        else:
            print("O Explorador subiu em {}."
                  .format("como ouro [+1000]!" if Ouro() not in self.things else "sem o ouro [+0]"))
        return True
    
    def step(self):
        agent = [thing for thing in self.things if isinstance(thing, Explorador)][0]
        for action in agent.programa(self.percept(agent)):
            self.execute_action(agent, action)


# Agente e o seu programa

E aqui definimos o próprio agente e o seu programa

In [29]:
from collections import deque

class Explorador(Agent):

    def __init__(self, dimencao):
        # Inicializa Agent com programa dummy; o nosso 'programa' é um método
        super().__init__(program=lambda p: None)
        self.dimrow = dimencao
        self.alive = True
        self.bump = False
        self.holding = []
        self.performance = 0
        self.has_arrow = True
        self.morto_por = ""
        self.t = 0
        self.location = (1, 1)
        self.direcao = 0  # 0: cima, 1: direita, 2: baixo, 3: esquerda

        
        self.kb = PropKB()

        
        self.visitadas = set()       # células já visitadas (confirmadas seguras)
        self.com_brisa = set()       # células onde se sentiu brisa
        self.com_fedor = set()       # células onde se sentiu fedor
        self.sem_brisa = set()       # células onde NÃO se sentiu brisa
        self.sem_fedor = set()       # células onde NÃO se sentiu fedor
        self.wumpus_morto = False
        self.fila_acoes = []         # fila de ações planeadas

       
        self.visitadas.add((1, 1))
        self._kb_tell_visitada(1, 1)
        self._kb_tell_segura(1, 1)

   

    def _prop(self, prefixo, x, y):
        """Cria o nome de uma proposição como string para uso na KB."""
        return expr(f'{prefixo}_{x}_{y}')

    def _kb_tell_segura(self, x, y):
        self.kb.tell(self._prop('OK', x, y))

    def _kb_tell_visitada(self, x, y):
        self.kb.tell(self._prop('V', x, y))

    def _kb_tell_brisa(self, x, y):
        self.kb.tell(self._prop('B', x, y))

    def _kb_tell_sem_brisa(self, x, y):
        self.kb.tell(self._prop('NB', x, y))

    def _kb_tell_fedor(self, x, y):
        self.kb.tell(self._prop('S', x, y))

    def _kb_tell_sem_fedor(self, x, y):
        self.kb.tell(self._prop('NS', x, y))

    def _kb_segura(self, x, y):
        return self.kb.ask_if_true(self._prop('OK', x, y))

  

    def _inferir_seguras(self):
        """Recalcula o conjunto de células seguras com base em toda a KB.

        Uma célula adjacente (ax, ay) é segura se existir PELO MENOS UMA
        célula visitada vizinha de (ax, ay) que NÃO tenha brisa E
        NÃO tenha fedor.

        Uma célula que já foi visitada é sempre segura.
        """
        seguras = set(self.visitadas) 

        
        for ax in range(1, self.dimrow + 1):
            for ay in range(1, self.dimrow + 1):
                if (ax, ay) in self.visitadas:
                    continue
                for (vx, vy) in self._adjacentes(ax, ay):
                    if (vx, vy) in self.visitadas:
                        if (vx, vy) in self.sem_brisa and (vx, vy) in self.sem_fedor:
                            seguras.add((ax, ay))
                            self._kb_tell_segura(ax, ay)
                            break
        return seguras



    def quadrado_a_seguir(self, local):
        """Devolve a célula à frente do agente a partir de 'local'."""
        if self.direcao == 0:
            return (local[0], local[1] + 1)
        if self.direcao == 1:
            return (local[0] + 1, local[1])
        if self.direcao == 2:
            return (local[0], local[1] - 1)
        if self.direcao == 3:
            return (local[0] - 1, local[1])

    def atualiza_loc_dir(self, acao):
        if acao == 'frente':
            self.location = self.quadrado_a_seguir(self.location)
        if acao == 'esquerda':
            self.direcao = (self.direcao - 1) % 4
        if acao == 'direita':
            self.direcao = (self.direcao + 1) % 4

    def executa_acao(self, acao):
        self.atualiza_loc_dir(acao)
        # Nota: o pega é tratado pelo ambiente (remove Ouro do ambiente e adiciona ao holding)

    def _adjacentes(self, x, y):
        """Devolve as células adjacentes (não diagonais) dentro dos limites."""
        candidatos = [(x - 1, y), (x + 1, y), (x, y - 1), (x, y + 1)]
        return [(cx, cy) for (cx, cy) in candidatos
                if 1 <= cx <= self.dimrow and 1 <= cy <= self.dimrow]

    def _bfs_caminho(self, inicio, destino, permitidas):
        """BFS: encontra o caminho mais curto de 'inicio' a 'destino'
        passando apenas por células em 'permitidas'.
        Devolve lista de células (sem incluir o início) ou None."""
        if inicio == destino:
            return []
        fila = deque([[inicio]])
        vistas = {inicio}
        while fila:
            caminho = fila.popleft()
            atual = caminho[-1]
            for viz in self._adjacentes(*atual):
                if viz not in vistas and viz in permitidas:
                    novo_caminho = caminho + [viz]
                    if viz == destino:
                        return novo_caminho[1:]
                    vistas.add(viz)
                    fila.append(novo_caminho)
        return None

    def _caminho_para_acoes(self, caminho):
        """Converte uma lista de células num plano de ações."""
        delta_para_dir = {
            (0, 1): 0,
            (1, 0): 1,
            (0, -1): 2,
            (-1, 0): 3
        }
        acoes = []
        dir_atual = self.direcao
        pos_atual = self.location
        for proxima in caminho:
            dx = proxima[0] - pos_atual[0]
            dy = proxima[1] - pos_atual[1]
            dir_alvo = delta_para_dir[(dx, dy)]
            diff = (dir_alvo - dir_atual) % 4
            if diff == 1:
                acoes.append('direita')
            elif diff == 2:
                acoes += ['direita', 'direita']
            elif diff == 3:
                acoes.append('esquerda')
            acoes.append('frente')
            dir_atual = dir_alvo
            pos_atual = proxima
        return acoes



    def _localizar_wumpus(self):
        """Tenta inferir a localização certa do Wumpus.
        Se a interseção dos adjacentes não visitados de todas as células com
        fedor resultar numa única célula, o Wumpus está lá com certeza."""
        if self.wumpus_morto or not self.com_fedor:
            return None
        candidatos = None
        for (fx, fy) in self.com_fedor:
            adj_nao_visitados = set(self._adjacentes(fx, fy)) - self.visitadas
            candidatos = adj_nao_visitados if candidatos is None else candidatos & adj_nao_visitados
        if candidatos and len(candidatos) == 1:
            return list(candidatos)[0]
        return None

    def _acoes_para_disparar(self, alvo):
        """Gera as ações para orientar o agente na direção do alvo e disparar."""
        dx = alvo[0] - self.location[0]
        dy = alvo[1] - self.location[1]
        delta_para_dir = {(0, 1): 0, (1, 0): 1, (0, -1): 2, (-1, 0): 3}
        dir_alvo = delta_para_dir.get((dx, dy))
        if dir_alvo is None:
            return []
        acoes = []
        diff = (dir_alvo - self.direcao) % 4
        if diff == 1:
            acoes.append('direita')
        elif diff == 2:
            acoes += ['direita', 'direita']
        elif diff == 3:
            acoes.append('esquerda')
        acoes.append('dispara')
        return acoes



    def _atualiza_kb(self, percecao):
        """Processa a perceção atual e atualiza a KB e os conjuntos internos."""
        (x, y) = self.location

        self.visitadas.add((x, y))
        self._kb_tell_visitada(x, y)
        self._kb_tell_segura(x, y)

        percecao_aqui = percecao[4]
        tipos_aqui = [type(p) for p in percecao_aqui]

        if Brisa in tipos_aqui:
            self.com_brisa.add((x, y))
            self._kb_tell_brisa(x, y)
        else:
            self.sem_brisa.add((x, y))
            self._kb_tell_sem_brisa(x, y)


        if Fedor in tipos_aqui:
            self.com_fedor.add((x, y))
            self._kb_tell_fedor(x, y)
        else:
            self.sem_fedor.add((x, y))
            self._kb_tell_sem_fedor(x, y)

        if Grito in tipos_aqui:
            self.wumpus_morto = True


    def programa(self, percecao):
        """Decide as ações a tomar com base na perceção e na KB."""
        print(f'Passo {self.t} | Pos={self.location} Dir={self.direcao} | ')
        self.t += 1

        if self.fila_acoes:
            proxima = self.fila_acoes.pop(0)
            print(f'  → (fila) {proxima}')
            return [proxima]

        self._atualiza_kb(percecao)


        seguras = self._inferir_seguras()

        (x, y) = self.location
        percecao_aqui = percecao[4]
        tipos_aqui = [type(p) for p in percecao_aqui]

        if Brilho in tipos_aqui:
            print('  → Ouro encontrado! A pegar...')
            return ['pega']


        if len(self.holding) > 0:
            if self.location == (1, 1):
                print('  → Em (1,1) com ouro! A sair...')
                return ['sair']
            print('  → Tem ouro! A navegar para (1,1)...')
            caminho = self._bfs_caminho(self.location, (1, 1), self.visitadas)
            if caminho:
                acoes = self._caminho_para_acoes(caminho)
                if acoes:
                    self.fila_acoes = acoes[1:]
                    return [acoes[0]]
            return ['sair']


        if self.has_arrow and self.com_fedor:
            alvo_wumpus = self._localizar_wumpus()
            if alvo_wumpus:
                adj = self._adjacentes(x, y)
                if alvo_wumpus in adj:
                    print(f'  → Wumpus localizado em {alvo_wumpus}! A disparar...')
                    acoes = self._acoes_para_disparar(alvo_wumpus)
                    if acoes:
                        self.fila_acoes = acoes[1:]
                        return [acoes[0]]
                else:
                    celulas_tiro = [c for c in self._adjacentes(*alvo_wumpus)
                                    if c in seguras and c != alvo_wumpus]
                    if celulas_tiro:
                        caminho = self._bfs_caminho(self.location, celulas_tiro[0], seguras)
                        if caminho:
                            acoes = self._caminho_para_acoes(caminho)
                            if acoes:
                                self.fila_acoes = acoes[1:]
                                return [acoes[0]]


        nao_visitadas_seguras = seguras - self.visitadas
        if nao_visitadas_seguras:
            melhor_caminho = None
            for destino in nao_visitadas_seguras:
                caminho = self._bfs_caminho(self.location, destino, seguras)
                if caminho is not None:
                    if melhor_caminho is None or len(caminho) < len(melhor_caminho):
                        melhor_caminho = caminho
                        melhor_destino = destino
            if melhor_caminho is not None:
                print(f'  → A explorar célula segura {melhor_destino}...')
                acoes = self._caminho_para_acoes(melhor_caminho)
                if acoes:
                    self.fila_acoes = acoes[1:]
                    return [acoes[0]]

        print('  → Sem células seguras para explorar. A voltar para (1,1)...')
        if self.location == (1, 1):
            return ['sair']
        caminho = self._bfs_caminho(self.location, (1, 1), self.visitadas)
        if caminho:
            acoes = self._caminho_para_acoes(caminho)
            if acoes:
                self.fila_acoes = acoes[1:]
                return [acoes[0]]
        return ['sair']


## Criação do agente e do mundo

Vamos então criar o mundo e po-lo a correr no tempo.

In [30]:
random.seed(1234567)

global grid, w
dimensao = 10
explorador = Explorador(dimensao)
w = WumpusEnvironment(explorador, dimensao)
grid = BlockGrid(w.width, w.height, fill=(0, 0, 0))

def draw_grid(world):
    color = {"Brisa": (128, 128, 128),
        "Poco": (128,128,128),
        "Ouro": (255, 255, 0),
        "Brilho": (255, 255, 0),
        "Wumpus": (255, 0, 0),
        "Fedor": (255, 0, 0),
        "Explorador": (0, 0, 255),
        "Parede": (255, 255, 255)
        }
    grid[:] = (0, 0, 0)
    for y in range(0,len(world)):
        for x in range(0,len(world[y])):
            if len(world[y][x]):
                grid[w.height-1-y, x] = color[world[y][x][-1].__class__.__name__]



passo = 0
max_passos = 500  
while not w.is_done() and passo < max_passos:
    clear_output(wait=True)
    draw_grid(w.get_world(True))
    grid.show()
    w.step()
    passo += 1
    sleep(0.3)


clear_output(wait=True)
draw_grid(w.get_world(True))
grid.show()
print(f'Passos executados: {passo}')
print('A performance do explorador foi: {}'.format(explorador.performance))

,,,,,,,,,,,
,,,,,,,,,,,
,,,,,,,,,,,
,,,,,,,,,,,
,,,,,,,,,,,
,,,,,,,,,,,
,,,,,,,,,,,
,,,,,,,,,,,
,,,,,,,,,,,
,,,,,,,,,,,
,,,,,,,,,,,


Passo 10 | Pos=(4, 4) Dir=2 | 
  → (fila) direita


KeyboardInterrupt: 